checking for virtual env path

In [1]:
!which python3

/home/rik07/Documents/repos/rag/ingestion/venv/bin/python3


document loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="/home/rik07/Documents/repos/rag/ingestion/datasets/attention.pdf")

documents = loader.load() # returns a list of document page
# list[Document] if .load() is called
# Iterable[Document] if .lazy_load() is called
print(documents[0].page_content[:500])

/home/rik07/Documents/repos/rag/ingestion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables andﬁgures in this paper solely for use in journalistic or
scholarly works.
Attention Is All Y ou Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez ∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


document splitter using langchain

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

doc_chunks = text_splitter.split_documents(documents=documents)
# .split_documents -> list[Document]

print(len(doc_chunks))
print(doc_chunks)

52
[Document(metadata={'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': 'Mozilla Firefox 147.0.3', 'creationdate': '2026-03-03T15:59:53+05:30', 'source': '/home/rik07/Documents/repos/rag/ingestion/datasets/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables andﬁgures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All Y ou Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez ∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurr

generating vector embeddings using embedding model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-small-en-v1.5")

texts = [doc.page_content for doc in doc_chunks]

doc_embeddings = embeddings.embed_documents(texts=texts)

print(f"document embeddings: {len(doc_embeddings)}")
print(f"vector dimension: {len(doc_embeddings[0])}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1165.05it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


document embeddings: 52
vector dimension: 384


storing vector embeddings into vector store

In [5]:
from langchain_chroma import Chroma

vector_db = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="../db_index"
)

vector_db.add_texts(
    texts=texts,
    embeddings=doc_embeddings
)

['a9c3026b-1543-4c1b-8e04-8980c12e75c4',
 '41356de5-18df-403d-86d9-4d0abb755e41',
 '1e1b4972-8fda-4470-a374-32d3181f01d1',
 'e905d0c2-5fa3-4502-8875-ec944ece449d',
 'e7b7ec39-8bcc-4499-ab17-870709b09478',
 'd06c40f3-1cc9-48b5-93ea-9c590f1473c9',
 '76bddec5-af72-4cd8-9dce-cfd130d698f0',
 '13e20f68-af9a-4413-b9f8-c7e23556a666',
 'afa2811d-ca09-40c4-9ae7-ada697acb927',
 '1210b10b-c322-4bf4-99d7-2fceeaff3a2d',
 '2bf3441e-d069-48d1-b0af-345869a9eb02',
 '49194302-e511-41b7-97c2-eb2a6055adaf',
 'e07a9627-d0d0-42a3-9fd6-963dc4b69120',
 'bfcc0b74-b157-478e-b984-050715d1a829',
 '6ddfd81b-6090-417d-af5f-68bcd93431f7',
 '2e8a2858-6b2e-463a-86a5-04c26340250f',
 'c17d64a9-30a4-41a7-80c0-9cc3451d4fd5',
 'd8854f83-71eb-457e-b50f-8b91eb9d7a40',
 '11c1c766-7477-4166-a21b-af5a9baf509e',
 'b7357c83-67ad-435d-b473-70c22fc2da24',
 '0d1d84bf-62f3-45c9-9beb-191a0461ddc5',
 '79830b86-498e-4bef-b7a2-6b07304d8f15',
 '0ae8c3dc-39ca-420e-9220-7a5a93888e31',
 '8bce3379-ab58-4ec1-a266-d5cc6ad2437b',
 'a1f93459-1096-

In [6]:
query = "What is tthe main topic of these documents ?"

docs = vector_db.similarity_search(query=query, k=2)

for doc in docs:
    print(f"---chunk content---\n{doc.page_content}")

---chunk content---
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated
corpus of english: The penn treebank.Computational linguistics, 19(2):313–330, 1993.
[26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In
Proceedings of the Human Language Technology Conference of the NAACL, Main Conference,
pages 152–159. ACL, June 2006.
[27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
model. InEmpirical Methods in Natural Language Processing, 2016.
[28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
summarization.arXiv preprint arXiv:1705.04304, 2017.
[29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact,
and interpretable tree annotation. InProceedings of the 21st International Conference on
Computational Linguistics and 44th Annual Meeting of the ACL, pages 433–440. ACL, July
2006.
